In [ ]:
%pip install -q rouge-score pandas

In [ ]:
import os
import pandas as pd
from rouge_score import rouge_scorer

In [ ]:
import re


def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF"
        u"\U0001FA00-\U0001FA6F"
        u"\U00002500-\U00002BFF"
        u"\U00002600-\U000026FF"
        u"✀-➿"
        u"⌀-⏿"
        "✓✗⚠►▸▼▲◄→←↑↓│├└┌┐┘┤┬┴┼"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', text)


def replace_arrows(text):
    if not isinstance(text, str):
        return ''
    arrows = [
        ('-->', ' leads to '),
        ('=>',  ' leads to '),
        ('→',   ' leads to '),
        ('►',   ' leads to '),
        ('▼',   ' then '),
        ('▲',   ' then '),
        ('←',   ' comes from '),
    ]
    for arrow, replacement in arrows:
        text = text.replace(arrow, replacement)
    return text


def extract_code_block_content(text):
    def clean_block(match):
        code = match.group(1)
        code = ''.join(
            ' ' if (0x2500 <= ord(ch) <= 0x257F) else ch
            for ch in code
        )
        code = re.sub(r'[ \t]+', ' ', code)
        code = re.sub(r'\n{2,}', '\n', code)
        return code.strip()

    return re.sub(
        r'```(?:\w+)?\n?(.*?)```',
        clean_block,
        text,
        flags=re.DOTALL
    )


def convert_table_to_text_safe(text):
    lines = text.split('\n')
    result = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if line.startswith('|') and line.endswith('|'):
            if re.match(r'^\|[\s\-:]+\|$', line):
                i += 1
                continue

            cells = [c.strip() for c in line.strip('|').split('|') if c.strip()]

            if i + 1 < len(lines) and re.match(r'^\|[\s\-:|]+\|$', lines[i + 1].strip()):
                headers = cells
                i += 2

                while i < len(lines):
                    row = lines[i].strip()
                    if not (row.startswith('|') and row.endswith('|')):
                        break

                    values = [c.strip() for c in row.strip('|').split('|') if c.strip()]
                    headers = [re.sub(r'\*+(.+?)\*+', r'\1', h) for h in headers]
                    values  = [re.sub(r'\*+(.+?)\*+', r'\1', v) for v in values]

                    pairs = []
                    for h, v in zip(headers, values):
                        if h and v:
                            pairs.append(f"{h} is {v}")

                    if pairs:
                        result.append('. '.join(pairs) + '.')

                    i += 1
            else:
                result.append('. '.join(cells) + '.')
                i += 1
        else:
            result.append(lines[i])
            i += 1

    return '\n'.join(result)


def ensure_line_terminators(text):
    lines = text.split('\n')
    result = []
    for line in lines:
        stripped = line.strip()
        match = re.match(r'^(\-|\*|•|\d+[\.\)])\s+(.+)', stripped)
        if match:
            stripped = match.group(2).strip()
        if stripped and stripped[-1] not in '.!?:':
            stripped += '.'
        result.append(stripped)
    return '\n'.join(result)


def ensure_sentence_terminators(text):
    paragraphs = text.split('\n')
    result = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            result.append('')
            continue
        doc = nlp(para)
        sentences = []
        for sent in doc.sents:
            s = sent.text.strip()
            if s and s[-1] not in '.!?:':
                s += '.'
            sentences.append(s)
        result.append(' '.join(sentences))
    return '\n'.join(result)

def remove_markdown_dividers(text):
    return re.sub(r'^\s*[-_*]{2,}\s*$', '', text, flags=re.MULTILINE)


def clean_markdown(text):

    text = replace_arrows(text)
    text = remove_emoji(text)

    text = extract_code_block_content(text)

    text = convert_table_to_text_safe(text)

    text = remove_markdown_dividers(text)

    text = re.sub(r'\*{1,3}(.+?)\*{1,3}', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    text = re.sub(r'\[(.+?)\]\(.+?\)', r'\1', text)
    text = re.sub(r'^[\-\*\+]\s+', '', text, flags=re.MULTILINE)

   
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")
    text = re.sub(r'"{2,}', '"', text)
    text = text.replace('"', '')
    text = text.replace('—', ' ')
    text = text.replace('–', ' ')
    text = re.sub(r'[ \t]+', ' ', text)  

    text = re.sub(r'#+\s*', '', text)
   
    return text.strip()

In [ ]:
INPUT_DIR  = "../Dataset/responses"
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Working dir:', os.getcwd())
print('Data dir   :', INPUT_DIR)
print('Output dir :', OUTPUT_DIR)

In [ ]:
def run_rouge_for_file(file_path, output_path,
                        reference_level='High Literacy',
                        compare_levels=None):
    
    df = pd.read_csv(file_path, sep='\t')
    df['LLM_Response'] = df['LLM_Response'].astype(str)
    df['LLM_Response'] = df['LLM_Response'].apply(clean_markdown)

    model_col = 'Model_Name' if 'Model_Name' in df.columns else 'Answer_Model'
    if compare_levels is None:
        compare_levels = ['Mid Literacy', 'Low Literacy', 'High Literacy']

    print(f"Loaded {len(df)} rows from {file_path}")
    print(f"   Models: {df[model_col].unique()}")
    print(f"   Levels: {df['Literacy_Level'].unique()}")

    scorer = rouge_scorer.RougeScorer(['rougeLsum'], use_stemmer=True)
    results = []

    for model_name in df[model_col].unique():
        df_model = df[df[model_col] == model_name]

        hl_df = (df_model[df_model['Literacy_Level'] == reference_level]
                 [['ID', 'LLM_Response']]
                 .rename(columns={'LLM_Response': 'HL_Text'}))

        if hl_df.empty:
            print(f"No {reference_level} rows for model: {model_name} — skipping")
            continue

        for level in compare_levels:
            comp_df = (df_model[df_model['Literacy_Level'] == level]
                       [['ID', 'LLM_Response']]
                       .rename(columns={'LLM_Response': 'Comp_Text'}))

            pairs = pd.merge(hl_df, comp_df, on='ID')

            if pairs.empty:
                print(f"No matched pairs for {model_name} | {level} — skipping")
                continue

            for _, row in pairs.iterrows():
                score = scorer.score(row['HL_Text'], row['Comp_Text'])
                results.append({
                    model_col:             model_name,
                    'ID':                  row['ID'],
                    'Compared_Level':      level,
                    'ROUGELsum_Precision': round(score['rougeLsum'].precision, 4),
                    'ROUGELsum_Recall':    round(score['rougeLsum'].recall,    4),
                    'ROUGELsum_F1':        round(score['rougeLsum'].fmeasure,  4)
                })

            print(f"    {model_name} | {level} — done")

    rouge_df = pd.DataFrame(results)
    rouge_df.to_csv(output_path, sep='\t', index=False)
    print(f"\n Saved: {output_path}")

    summary = rouge_df.groupby([model_col, 'Compared_Level']).agg(
        Pairs         = ('ROUGELsum_F1',        'count'),
        Avg_Precision = ('ROUGELsum_Precision', 'mean'),
        Avg_Recall    = ('ROUGELsum_Recall',    'mean'),
        Avg_F1        = ('ROUGELsum_F1',        'mean')
    ).round(4)

    summary = summary.reindex(compare_levels, level='Compared_Level')

    print(f"\n{'='*60}")
    print(f"  ROUGE-Lsum: {reference_level} vs Other Levels")
    print(f"{'='*60}")
    print(summary.to_string())

    return rouge_df

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_claude_explicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_gemini_explicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_gpt_explicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_llama_explicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_qwen_explicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_claude_implicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy', 'High Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_gemini_implicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy', 'High Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_gpt_implicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy', 'High Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_llama_implicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy', 'High Literacy'],
)

In [ ]:
run_rouge_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "ROUGELsum_qwen_implicit.tsv"),
    compare_levels = ['Mid Literacy', 'Low Literacy', 'High Literacy'],
)